In [5]:
%pip install duckdb pandas matplotlib seaborn plotly numpy openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
import duckdb

df_volunteers= pd.read_csv("../data/volunteers.csv")

df_interviews = pd.read_excel("../data/interviews.xlsx")

df_analytics = pd.read_excel("../data/analytics.xlsx")


## Connect with db + import csv files as datasets

In [31]:

con = duckdb.connect()

con.register("volunteers", df_volunteers)
con.register("interviews", df_interviews)
con.register("analytics", df_analytics)
con.execute("""
CREATE VIEW raw_volunteers AS
SELECT *
FROM read_csv_auto('../data/volunteers.csv');
""")

con.execute("""
CREATE VIEW raw_interviews AS
SELECT *
FROM read_csv_auto('../data/interviews.csv');
""")

con.execute("""
CREATE VIEW raw_analytics AS
SELECT *
FROM read_csv_auto('../data/analytics.csv');
""")



## Check datasets info

In [22]:
query = """
SELECT *
FROM raw_volunteers
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

                 Name                        Creation log  \
0          Mariam Ali  U+ HR Account Sep 11, 2025 1:30 PM   
1  Manusha Srikanthan     Jess Scott Sep 11, 2025 1:45 PM   
2          Harvi Shah     Jess Scott Sep 11, 2025 4:36 PM   
3      Dorsey Bangarh     Jess Scott Sep 11, 2025 5:37 PM   
4           MUZI CHEN     Jess Scott Sep 11, 2025 6:42 PM   
5       Pushti Ladani     Jess Scott Sep 11, 2025 7:26 PM   
6        Purti Ladani     Jess Scott Sep 11, 2025 7:29 PM   
7        Harjot Singh     Jess Scott Sep 11, 2025 8:52 PM   
8       Rincy Nahomie     Jess Scott Sep 11, 2025 8:31 PM   
9          JeeHu Choi     Jess Scott Sep 11, 2025 8:55 PM   

                             Email Address                       YRES Email  \
0              mariam.ali@yorkeducation.ca                              NaN   
1              manushasrikanthan@gmail.com                              NaN   
2                  harvishah2602@gmail.com      harvi.shah@yorkeducation.ca   
3           

In [25]:
query = """
SELECT *
FROM raw_interviews
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

     Invitee Name Invitee First Name Invitee Last Name  \
0     Yuhan Cheng              Yuhan             Cheng   
1       Justin Wu             Justin                Wu   
2       Justin Wu             Justin                Wu   
3   Sinaz Heidari              Sinaz           Heidari   
4     Nicole Zhou             Nicole              Zhou   
5       Amy Liang                Amy             Liang   
6    Isabella Lee           Isabella               Lee   
7     April Cheng              April             Cheng   
8    Anaya Surati              Anaya            Surati   
9  Dana Chowdhury               Dana         Chowdhury   

                  Invitee Email  
0           cyuhan203@gmail.com  
1  justin.wu.sparking@gmail.com  
2  justin.wu.sparking@gmail.com  
3    sinazheidari1380@gmail.com  
4       nicolezhou106@gmail.com  
5          amareliang@gmail.com  
6           ialee2329@gmail.com  
7      chengaprilqing@gmail.com  
8         anayasurati@gmail.com  
9            62410bwv

In [26]:
query = """
SELECT *
FROM raw_analytics
LIMIT 10;
"""
result = con.execute(query).df()
print(result)

                         Name               Display name  \
0   (Vol. Leader) Victoria H.  (Vol. Leader) Victoria H.   
1  (Vol. Leader) Zainab Ahmed                     Zainab   
2               Aadam Lakhani              Aadam Lakhani   
3               Aadhya Sriram              Aadhya Sriram   
4                   Aakanksha                  Aakanksha   
5                  Aali Vaqar                 Aali Vaqar   
6               Aanchal Ratha              Aanchal Ratha   
7          Aanushan Elangoban         Aanushan Elangoban   
8          Aanushan Elangoban         Aanushan Elangoban   
9      Aaradhya Atul Yeginwar     Aaradhya Atul Yeginwar   

                                 Email Account type Account created (UTC)  \
0                 iivv.berry@gmail.com       Member          Apr 14, 2023   
1        zainab.ahmed@yorkeducation.ca       Member          Apr 14, 2023   
2       aadam.lakhani@yorkeducation.ca       Member          Sep 16, 2023   
3       aadhya.sriram@yorkeduca

## Clean datasets

In [32]:
con.execute(r"""
CREATE OR REPLACE VIEW clean_volunteers AS
WITH base AS (
  SELECT
    TRIM(Name) AS full_name,

    -- 1) extract "Sep 26, 2025 3:07 AM" from the messy text
    regexp_extract(
      "Creation Log",
      '([A-Z][a-z]{2}\s+\d{1,2},\s+\d{4}\s+\d{1,2}:\d{2}\s+[AP]M)',
      1
    ) AS creation_str,

    LOWER(TRIM("Email Address")) AS email,
    LOWER(TRIM("YRES Email")) AS yres_email,
    TRIM("Youth Advisor") AS youth_advisor

  FROM raw_volunteers
)
SELECT
  full_name,

  -- 2) parse into TIMESTAMP (safe)
  TRY_STRPTIME(creation_str, '%b %d, %Y %I:%M %p') AS creation_ts,

  email,
  yres_email,
  youth_advisor
FROM base
WHERE full_name IS NOT NULL
""")

df_volunteers_clean = con.execute("""
SELECT *
FROM clean_volunteers
LIMIT 20
""").df()

df_volunteers_clean


,full_name,creation_ts,email,yres_email,youth_advisor
0,Mariam Ali,2025-09-11 13:30:00,mariam.ali@yorkeducation.ca,NaN,Mariam Ali
1,Manusha Srikanthan,2025-09-11 13:45:00,manushasrikanthan@gmail.com,NaN,NaN
2,Harvi Shah,2025-09-11 16:36:00,harvishah2602@gmail.com,harvi.shah@yorkeducation.ca,Tiffany Ye
3,Dorsey Bangarh,2025-09-11 17:37:00,dorseybangarh2004@gmail.com,dorsey.bangarh@yorkeducation.ca,NaN
4,MUZI CHEN,2025-09-11 18:42:00,muzi21@g.ucla.edu,muzi.chen@yorkeducation.ca,NaN
5,Pushti Ladani,2025-09-11 19:26:00,pushti-kantilal.ladani@mohawkcollege.ca,pushti.ladani@yorkeducation.ca,NaN
6,Purti Ladani,2025-09-11 19:29:00,purti1002@gmail.com,purti.ladani@yorkeducation.ca,NaN
7,Harjot Singh,2025-09-11 20:52:00,singhharjot1312@gmail.com,harjot.singh@yorkeducation.ca,Aanushan Elangoban
8,Rincy Nahomie,2025-09-11 20:31:00,rincynahomie85@gmail.com,rincy.nahomie@yorkeducation.ca,NaN
9,JeeHu Choi,2025-09-11 20:55:00,jiwhotwin@gmail.com,jeehu.choi@yorkeducation.ca,Abby Akinyemi


In [34]:
con.execute("""
CREATE OR REPLACE VIEW clean_interviews AS
WITH base AS (
    SELECT
        TRIM("Invitee Name") AS full_name,

        TRIM("Invitee First Name") AS first_name,

        TRIM("Invitee Last Name") AS last_name,

        LOWER(TRIM("Invitee Email")) AS email

    FROM raw_interviews
)
SELECT *
FROM base
WHERE email IS NOT NULL
""")
df_interviews_clean = con.execute("""
SELECT *
FROM clean_interviews
LIMIT 20
""").df()
df_interviews_clean

,full_name,first_name,last_name,email
0,Yuhan Cheng,Yuhan,Cheng,cyuhan203@gmail.com
1,Justin Wu,Justin,Wu,justin.wu.sparking@gmail.com
2,Justin Wu,Justin,Wu,justin.wu.sparking@gmail.com
3,Sinaz Heidari,Sinaz,Heidari,sinazheidari1380@gmail.com
4,Nicole Zhou,Nicole,Zhou,nicolezhou106@gmail.com
5,Amy Liang,Amy,Liang,amareliang@gmail.com
6,Isabella Lee,Isabella,Lee,ialee2329@gmail.com
7,April Cheng,April,Cheng,chengaprilqing@gmail.com
8,Anaya Surati,Anaya,Surati,anayasurati@gmail.com
9,Dana Chowdhury,Dana,Chowdhury,62410bwv@gmail.com


In [36]:
con.execute("""
CREATE OR REPLACE VIEW clean_analytics AS
WITH base AS (
    SELECT
        TRIM(Name) AS full_name,

        TRIM("Display name") AS display_name,

        LOWER(TRIM(Email)) AS email,

        TRIM("Account type") AS account_type,

        TRY_CAST("Account created (UTC)" AS TIMESTAMP) AS account_created,

        TRY_CAST("Claimed Date (UTC)" AS TIMESTAMP) AS claimed_date,

        TRY_CAST("Days active" AS INTEGER) AS days_active,

        TRY_CAST("Messages posted" AS INTEGER) AS messages_posted

    FROM raw_analytics
)
SELECT *
FROM base
WHERE email IS NOT NULL
""")
df_analytics_clean = con.execute("""
SELECT *
FROM clean_analytics
LIMIT 20
""").df()
df_analytics_clean


,full_name,display_name,email,account_type,account_created,claimed_date,days_active,messages_posted
0,(Vol. Leader) Victoria H.,(Vol. Leader) Victoria H.,iivv.berry@gmail.com,Member,NaT,NaT,0,0
1,(Vol. Leader) Zainab Ahmed,Zainab,zainab.ahmed@yorkeducation.ca,Member,NaT,NaT,4,3
2,Aadam Lakhani,Aadam Lakhani,aadam.lakhani@yorkeducation.ca,Member,NaT,NaT,0,0
3,Aadhya Sriram,Aadhya Sriram,aadhya.sriram@yorkeducation.ca,Member,NaT,NaT,0,0
4,Aakanksha,Aakanksha,aakanksha.patel@yorkeducation.ca,Member,NaT,NaT,5,0
5,Aali Vaqar,Aali Vaqar,aali.vaqarahmad@yorkeducation.ca,Member,NaT,NaT,0,0
6,Aanchal Ratha,Aanchal Ratha,aanchal.ratha@yorkeducation.ca,Member,NaT,NaT,0,0
7,Aanushan Elangoban,Aanushan Elangoban,aanushan.elangoban@yorkeducation.ca,Admin,NaT,NaT,30,394
8,Aanushan Elangoban,Aanushan Elangoban,aanushan.236@gmail.com,Admin,NaT,NaT,0,0
9,Aaradhya Atul Yeginwar,Aaradhya Atul Yeginwar,aaradhya@yorkeducation.ca,Member,NaT,NaT,0,0
